# Market basket — what gets bought together

This notebook does one thing: **find patterns in orders** so you can say things like “people who bought A often bought B”, and save numbers (**support**, **confidence**, **lift**) for your report.

**Data:** `order_items_dataset.csv` in this same folder. Each row is a line; `order_id` groups rows into one basket (one checkout).

**Website later:** we use **`product_id`** in baskets so each rule can point to a real row in your database.

**Libraries:** `pandas`, `mlxtend`. Install if needed:

`pip install pandas mlxtend`


## 1. Setup

Tweak `MIN_SUPPORT` if you get too few or too many rules. Lower support → more patterns (but noisier).

In [ ]:
from pathlib import Path

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

# --- paths: this folder first, then fall back to main project data ---
HERE = Path.cwd()
ORDER_CSV = HERE / "order_items_dataset.csv"
if not ORDER_CSV.exists():
    ORDER_CSV = HERE.parent / "data" / "market_basket" / "order_items_dataset.csv"

OUT_DIR = HERE / "outputs"
OUT_DIR.mkdir(exist_ok=True)

# --- mining knobs (change these if needed) ---
MIN_SUPPORT = 0.02          # share of baskets that contain the itemset (2%)
MIN_LIFT = 1.0              # keep rules with lift > 1 (better than random)
TOP_N_RULES = 40            # how many rows to show in the notebook
RECOMMEND_TOP_K = 8         # max recommendations per product (for export)

print("CSV:", ORDER_CSV.resolve())
print("Outputs:", OUT_DIR.resolve())

## 2. Load the CSV and quick checks

Nothing fancy — we just want to see row count, missing values, and how many orders we have.

In [ ]:
df = pd.read_csv(ORDER_CSV)
print("Shape:", df.shape)
print(df.head())
print("\nMissing:", df[["order_id", "product_id", "product_name"]].isna().sum().to_dict())
print("Unique orders:", df["order_id"].nunique())
print("Unique products:", df["product_id"].nunique())

## 3. Turn orders into "transactions"

One transaction = everything in one `order_id`. We use **`product_id` as string** so it matches your DB primary key and stays short for the ML step.

If the same product appears twice in one order, we keep it once (a set).

In [ ]:
df["pid"] = df["product_id"].astype(str)

transactions = (
    df.groupby("order_id")["pid"]
    .apply(lambda items: sorted(set(items)))
    .tolist()
)

sizes = [len(t) for t in transactions]
print(f"Transactions: {len(transactions)}")
print(f"Basket size — min: {min(sizes)}, max: {max(sizes)}, avg: {sum(sizes)/len(sizes):.2f}")
print("Example basket:", transactions[0])

## 4. Encode baskets → run FP-Growth → build rules

**FP-Growth** finds frequent itemsets (groups of products that show up often). Then **association rules** add confidence and lift.

- **Support:** how often the itemset appears in all baskets.
- **Confidence:** P(consequent | antecedent) — if someone bought the left side, how often they also bought the right side.
- **Lift:** how much more likely the consequent is compared to baseline (lift > 1 is usually what you want).

We use FP-Growth first; it’s usually faster than Apriori on bigger data.

In [ ]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

print("Encoded matrix shape:", basket_df.shape, "(rows = baskets, cols = distinct products)")

frequent = fpgrowth(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
print("Frequent itemsets found:", len(frequent))
if frequent.empty:
    raise ValueError("No frequent itemsets — lower MIN_SUPPORT (e.g. 0.01) and run again.")

rules = association_rules(frequent, metric="confidence", min_threshold=0.2)
rules = rules[rules["lift"] > MIN_LIFT].sort_values(["lift", "confidence"], ascending=False)
print("Rules after lift filter:", len(rules))
if rules.empty:
    raise ValueError("No rules passed the filters — lower MIN_SUPPORT or MIN_LIFT, or relax min_threshold in association_rules.")

rules.head(TOP_N_RULES)

## 5. Rules that are easy to use on a product page

On a product detail page you usually want: **one product on the left** → **recommended products on the right**.

So we keep rules where the antecedent is **exactly one product** (`frozenset` with one item). Rules with two products on the left need more context (cart), so we skip them for now.

In [ ]:
def frozenset_len(fs):
    return len(fs) if isinstance(fs, frozenset) else len(fs)

singleton = rules[rules["antecedents"].apply(frozenset_len) == 1].copy()
if singleton.empty:
    print("No single-item antecedent rules — try lowering MIN_SUPPORT or confidence min_threshold.")
else:
    print("Singleton-antecedent rules:", len(singleton))

singleton["antecedent_id"] = singleton["antecedents"].apply(lambda x: list(x)[0])
singleton["consequent_ids"] = singleton["consequents"].apply(lambda x: ", ".join(sorted(x)))

cols = [
    "antecedent_id",
    "consequent_ids",
    "antecedent support",
    "consequent support",
    "support",
    "confidence",
    "lift",
    "leverage",
    "conviction",
]
show = singleton[[c for c in cols if c in singleton.columns]]
show.head(TOP_N_RULES)

## 6. Export for your report and for the website

- `outputs/association_rules_singleton.csv` — human-readable rules (one product → others).
- `outputs/recommendations_top_k.csv` — for each antecedent product, up to **K** consequents ranked by lift (good starting point for “recommended products” in code).

Open the CSVs in Excel if you want to paste tables into the report.

In [ ]:
singleton.to_csv(OUT_DIR / "association_rules_singleton.csv", index=False)

# explode consequents: one row per (antecedent, single consequent item)
rows = []
for _, row in singleton.iterrows():
    ant = row["antecedent_id"]
    for cons in row["consequents"]:
        if str(cons) == str(ant):
            continue
        rows.append(
            {
                "antecedent_product_id": ant,
                "recommended_product_id": cons,
                "support": row["support"],
                "confidence": row["confidence"],
                "lift": row["lift"],
            }
        )

pair_df = pd.DataFrame(rows)
if not pair_df.empty:
    pair_df = pair_df.sort_values(["antecedent_product_id", "lift", "confidence"], ascending=[True, False, False])
    topk = pair_df.groupby("antecedent_product_id", as_index=False).head(RECOMMEND_TOP_K)
    topk.to_csv(OUT_DIR / "recommendations_top_k.csv", index=False)
    print("Saved:", OUT_DIR / "recommendations_top_k.csv")
else:
    print("No pair rows to export — try lowering MIN_SUPPORT or MIN_LIFT.")

print("Saved:", OUT_DIR / "association_rules_singleton.csv")

## 7. Optional: Apriori (same result idea, different algorithm)

Apriori does the same job as FP-Growth for frequent itemsets; it can be slower when you have many products. Uncomment and run if your teacher wants both algorithms named in the report.

If this runs for a long time, bump `MIN_SUPPORT` up a bit.

In [ ]:
# frequent_ap = apriori(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
# rules_ap = association_rules(frequent_ap, metric="confidence", min_threshold=0.2)
# rules_ap = rules_ap[rules_ap["lift"] > MIN_LIFT].sort_values(["lift", "confidence"], ascending=False)
# print("Apriori rules (count):", len(rules_ap))
# rules_ap.head(15)
print("Uncomment the lines above to run Apriori.")

## 8. How to write this up (short)

Pick **3–5 rules** with high lift and sensible confidence. For each, say in plain language what it means for the shop (bundles, shelf placement, “customers also bought” on the site).

If support is very small, say the pattern is **weak** — you’d want more orders before trusting it in production.